##### [ CNN기반 CiFa10 이미지 분류 모델 ]

- 데이터셋 : Pytorch 내장 데이터셋 활용
- 학습종류 : 지도학습 + 다중분류


In [ ]:
## 모듈 로딩
import torch                                    # 텐서
import torch.nn as nn                           # 인공신경망
import torch.nn.functional as F                 # 인공신경망함수

import torchvision.transforms as transforms     # 이미지 전처리 변형
from torchvision.datasets import CIFAR10         # 내장 데이터 셋
from torch.utils.data import DataLoader         # 학습 데이터 로딩 관련

import numpy as np                                 # 데이터 저장 형식 관련 모듈
import matplotlib.pyplot as plt                 #   이미지 시각화 모듈

In [4]:
## 준비 ==> 전처리용 transforms 인스턴ㅅ, 저장위치

ROOT = '../_data/image/'

import os
if not os.path.exists(ROOT):
    os.makedirs(ROOT)
else:
    print(f'{ROOT}: 존재함')

../_data/image/: 존재함


In [5]:
print(CIFAR10.meta)
print(CIFAR10.test_list)
for a in CIFAR10.train_list:
    print(a)


{'filename': 'batches.meta', 'key': 'label_names', 'md5': '5ff9c542aee3614f3951f8cda6e48888'}
[['test_batch', '40351d587109b95175f43aff81a1287e']]
['data_batch_1', 'c99cafc152244af753f735de768cd75f']
['data_batch_2', 'd4bba439e000b95fd0a9bffe97cbabec']
['data_batch_3', '54ebc095f3ab1f0389bbae665268c751']
['data_batch_4', '634d18415352ddfa80567beed471001a']
['data_batch_5', '482c414d41f54cd18b22e5b47cb7c3cb']


In [ ]:
data = CIFAR10(ROOT, download=True)


Files already downloaded and verified


In [67]:
from sklearn.model_selection import train_test_split
from torch.utils.data import random_split

In [ ]:

# 전처리 (예시)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(32),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=(-0.15, 0.15)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    
])
# 학습용 데이터셋
trainDS = CIFAR10(root=ROOT, train=True, download=True, transform=transform)

# 테스트용 데이터셋
testDS = CIFAR10(root=ROOT, train=False, download=True, transform=transform)


Files already downloaded and verified
Files already downloaded and verified


In [69]:
len(trainDS)

50000

In [71]:
generator = torch.Generator().manual_seed(42)
trainDS, validDS = random_split(trainDS, [45000, 5000], generator=generator)


In [38]:
BATCH_SIZE = 1000


In [73]:
trainDL = DataLoader(trainDS, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validDL = DataLoader(validDS, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

testDL = DataLoader(testDS, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

In [74]:
for a, b in trainDL:
    print(a,b)
    break

tensor([[[[ 0.3994,  0.6221,  0.8276,  ..., -1.5699, -1.8268, -1.8953],
          [ 0.0569,  0.1597,  0.2111,  ..., -1.8097, -1.8782, -1.6384],
          [-0.1486, -0.0116,  0.0056,  ..., -1.9809, -1.9467, -1.0219],
          ...,
          [-0.8335, -1.0048, -0.9020,  ..., -1.1247, -1.1760, -1.0562],
          [-0.8335, -0.9534, -0.8335,  ..., -0.5767, -1.0733, -1.2959],
          [-0.7993, -0.9877, -0.9020,  ...,  0.3309,  0.0056, -0.7479]],

         [[-1.3704, -1.0553, -0.9153,  ..., -1.6681, -1.8782, -1.9482],
          [-0.3725, -0.0924, -0.0749,  ..., -1.8606, -1.9482, -1.8081],
          [-0.2325, -0.0749,  0.0651,  ..., -1.9132, -1.9132, -1.2654],
          ...,
          [-1.5980, -1.7731, -1.6856,  ..., -2.0007, -1.9832, -1.9657],
          [-1.6155, -1.7381, -1.6506,  ..., -1.8431, -1.9657, -2.0007],
          [-1.5980, -1.7381, -1.7031,  ..., -1.3354, -1.5105, -1.8606]],

         [[-1.6824, -1.4733, -1.2990,  ..., -1.7173, -1.7522, -1.7173],
          [-0.7238, -0.4798, -

In [75]:
## 데이터셋 체크
## - 타입
print(f'type         : {type(testDS)}')

## - 속성 : 클래스 정보
print(f'classes      : {testDS.classes}')
print(f'class_to_idx : {testDS.class_to_idx}')

type         : <class 'torchvision.datasets.cifar.CIFAR10'>
classes      : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
class_to_idx : {'airplane': 0, 'automobile': 1, 'bird': 2, 'cat': 3, 'deer': 4, 'dog': 5, 'frog': 6, 'horse': 7, 'ship': 8, 'truck': 9}


In [61]:
## - 속성 : 데이터와 타겟
print(f'targets      : {testDS.targets} {len(trainDS.classes)}')
print(f'data         : {testDS.data.shape}')


targets      : [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9, 5, 2, 4, 0, 9, 6, 6, 5, 4, 5, 9, 2, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9, 7, 6, 9, 8, 0, 3, 8, 8, 7, 7, 4, 6, 7, 3, 6, 3, 6, 2, 1, 2, 3, 7, 2, 6, 8, 8, 0, 2, 9, 3, 3, 8, 8, 1, 1, 7, 2, 5, 2, 7, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 6, 2, 1, 3, 0, 4, 2, 7, 8, 3, 1, 2, 8, 0, 8, 3, 5, 2, 4, 1, 8, 9, 1, 2, 9, 7, 2, 9, 6, 5, 6, 3, 8, 7, 6, 2, 5, 2, 8, 9, 6, 0, 0, 5, 2, 9, 5, 4, 2, 1, 6, 6, 8, 4, 8, 4, 5, 0, 9, 9, 9, 8, 9, 9, 3, 7, 5, 0, 0, 5, 2, 2, 3, 8, 6, 3, 4, 0, 5, 8, 0, 1, 7, 2, 8, 8, 7, 8, 5, 1, 8, 7, 1, 3, 0, 5, 7, 9, 7, 4, 5, 9, 8, 0, 7, 9, 8, 2, 7, 6, 9, 4, 3, 9, 6, 4, 7, 6, 5, 1, 5, 8, 8, 0, 4, 0, 5, 5, 1, 1, 8, 9, 0, 3, 1, 9, 2, 2, 5, 3, 9, 9, 4, 0, 3, 0, 0, 9, 8, 1, 5, 7, 0, 8, 2, 4, 7, 0, 2, 3, 6, 3, 8, 5, 0, 3, 4, 3, 9, 0, 6, 1, 0, 9, 1, 0, 7, 9, 1, 2, 6, 9, 3, 4, 6, 0, 0, 6, 6, 6, 3, 2, 6, 1, 8, 2, 1, 6, 8, 6, 8, 0, 4, 0, 7, 7, 5, 5, 3, 5, 2, 3, 4, 1, 7, 5, 

[3] 모델 정의 및 설계 <hr>

In [ ]:

class cf_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        ## 특징맵 추출 부분
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # (B, 3, 32, 32) -> (B, 16, 32, 32)
        self.pool1 = nn.MaxPool2d(2, 2)                          # (B, 16, 32, 32) -> (B, 16, 25, 25)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # (B, 16, 25, 25) -> (B, 32, 25, 25)
        self.pool2 = nn.MaxPool2d(2, 2)                          # (B, 32, 25, 25) -> (B, 32, 12, 12)
        
        self.flatten = nn.Flatten()  # (B, 32, 12, 12) -> (B, 32 * 12 * 12)

        ## 전결합 학습 부분
        self.fc1 = nn.Linear(3072,512)  # 3072 -> 512
        self.dropout = nn.Dropout(0.25)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.25)
        self.fc4 = nn.Linear(64, 32)
        self.out = nn.Linear(32, 10)  

        

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))

        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = F.relu(self.fc4(x))
        x = self.out(x)  

        return x  

In [78]:
num_epochs = 5
total_batch = len(trainDL)
print('총 배치의 수 : {}'.format(total_batch))

총 배치의 수 : 45


In [82]:
MODEL = cf_CNN()

USE_CUDA = torch.cuda.is_available()
device = torch.device("cuda" if USE_CUDA else "cpu")
MODEL.to(device)

cf_CNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=3072, out_features=512, bias=True)
  (dropout): Dropout(p=0.25, inplace=False)
  (fc2): Linear(in_features=512, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (fc4): Linear(in_features=64, out_features=32, bias=True)
  (out): Linear(in_features=32, out_features=1, bias=True)
)

In [ ]:
LOSS_FN = nn.CrossEntropyLoss()
OPTIMIZER = torch.optim.Adam(MODEL.parameters(), lr=0.001)

In [97]:
def training(dataloader):
    ## 학습 모드 설정
    MODEL.train()

    ## 학습 손실과 점수 저장 
    total_loss, total_acc = 0, 0
    
    for idx, (text, label) in enumerate(dataloader):

        OPTIMIZER.zero_grad()
        pre  = MODEL(text)

        loss = LOSS_FN(pre, label)
        loss.backward()
        ## gradient vanishing, gradient exploding 발생 => 방지 및 안정화 
        ## - gradient가 일정 threshold를 넘어가면 clipping
        ## - clipping: gradient의 L2norm(norm이지만 보통 L2 norm사용)으로 나눠주는 방식
        torch.nn.utils.clip_grad_norm_(MODEL.parameters(), 0.1)
        OPTIMIZER.step()

        total_loss += loss.item()
        total_acc += (pre.argmax(dim=1) == label).sum().item()

        if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1

In [98]:
def evaluate(dataloader):
    MODEL.eval()
    total_loss, total_acc = 0, 0

    with torch.no_grad():
        for idx, (text, label) in enumerate(dataloader):
            ## 추론 진행
            pre = MODEL(text.float())
            ## 손실 계산
            loss = LOSS_FN(pre, label.reshape(-1,1).float())

            ## 손실 및 성능 평가
            total_loss += loss.item()
            total_acc += (pre.argmax(1) == label).sum().item()
            if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1

In [105]:
for idx, (text, label) in enumerate(trainDL):
    print(text.shape, label.shape)
    break

torch.Size([1000, 3, 32, 32]) torch.Size([1000])


In [100]:
## 모델 및 모델 층별 상태값 즉, 파라미터 값 저장 경로
MODEL_DIR  = './models/'
MODEL_FILE = 'IMDB_RNN_MODEL.pt'
EPOCHS = 100  ## 임시
# 모델 저장 기준
MAX_ACC = 0.

for epoch in range(1, EPOCHS + 1):
    
    train_loss, train_acc = training(trainDL)
    valid_loss, valid_acc = evaluate(validDL)
    # SCHEDULER.step()

    print("-" * 59)
    print(f'| end of epoch {epoch:3d} | train acc {train_acc:8.3f}  | valid acc {valid_acc:8.3f}')
    print("-" * 59)

    ## 모델 저장 
    if MAX_ACC < valid_acc : 
        torch.save(MODEL, MODEL_DIR+MODEL_FILE)
        MAX_ACC = valid_acc


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1000x2048 and 3072x512)